<a href="https://colab.research.google.com/github/tranmailinheecneu-arch/bank-failure-pipeline/blob/main/part3_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==4.0.1 polars psutil pyarrow
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from google.colab import files
up = files.upload()   # spark_input.parquet AND part3_polars_pandas.py
print(list(up))

Saving part3_polars_pandas.py to part3_polars_pandas.py
Saving spark_input.parquet to spark_input.parquet
['part3_polars_pandas.py', 'spark_input.parquet']


In [ ]:
import os, subprocess, psutil
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True)
print("RAM total (GB):", round(psutil.virtual_memory().total / 1e9, 1))
print("JAVA_HOME now:", os.environ.get("JAVA_HOME"))
print("--- /usr/lib/jvm ---"); print(sh("ls /usr/lib/jvm").stdout or sh("ls /usr/lib/jvm").stderr)
print("--- java -version ---"); r = sh("java -version"); print(r.stdout + r.stderr)

RAM total (GB): 13.6
JAVA_HOME now: /usr/lib/jvm/java-17-openjdk-amd64
--- /usr/lib/jvm ---
java-1.21.0-openjdk-amd64
java-21-openjdk-amd64

--- java -version ---
[0.014s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.014s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)



In [ ]:
r = sh("apt-get update -qq && apt-get install -y openjdk-17-jdk-headless")
print((r.stdout + r.stderr)[-1500:])
print(sh("ls /usr/lib/jvm").stdout)

database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.21.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.21.1+2-1~24.04.1) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.21.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.21.1+2-1~24.04.1) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.21.1+2-1~24.04.1) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to ha

In [ ]:
import os, sys, glob, subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True)

r = sh("apt-get update -qq && apt-get install -y openjdk-17-jdk-headless")
print((r.stdout + r.stderr)[-800:])

cand = sorted(glob.glob("/usr/lib/jvm/*17*")) or sorted(glob.glob("/usr/lib/jvm/*"))
print("java folders:", cand)
os.environ["JAVA_HOME"] = cand[0]
os.environ["PATH"] = cand[0] + "/bin:" + os.environ["PATH"]
r = sh("java -version"); print(r.stdout + r.stderr)
print("files in /content:"); print(sh("ls -lh /content").stdout)

Reading package lists...
Building dependency tree...
Reading state information...
openjdk-17-jdk-headless is already the newest version (17.0.21.1+2-1~24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 68 not upgraded.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

java folders: ['/usr/lib/jvm/java-1.17.0-openjdk-amd64', '/usr/lib/jvm/java-17-openjdk-amd64']
openjdk version "17.0.21.1" 2026-11-17
OpenJDK Runtime Environment (build 17.0.21.1+2-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.1+2-1-24.04.1-Ubuntu, mixed mode, sharing)

files in /content:
total 8.0K
-rw-r--r-- 1 root root 3.9K Nov 23 03:59 part3_polars_pandas.txt
drwxr-xr-x 1 root root 4.0K Nov 18 15:58 sample_data



In [ ]:
import os
print(os.listdir("/content"))
import io, os, re, sys, time, contextlib, platform
import numpy as np, pandas as pd, polars as pl, psutil
import pyspark
from pyspark.sql import SparkSession, DataFrame, Window, functions as F
from part3_polars_pandas import polars_pipeline, FEATURES, KEYS, VALUE_COLS

os.environ["PYSPARK_PYTHON"] = sys.executable      # use the notebook's own Python
PATH = "/content/spark_input.parquet"
log = []
def say(s=""):
    log.append(str(s)); print(s)

spark = (SparkSession.builder.master("local[*]")
         .config("spark.driver.memory", "5g")
         .config("spark.sql.shuffle.partitions", "16")      # default 200 is far too many tasks for 1.7M rows
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
say(f"machine: {os.cpu_count()} logical CPUs, {psutil.virtual_memory().total/1e9:.1f} GB RAM | "
    f"spark {spark.version} | pyspark {pyspark.__version__} | polars {pl.__version__} | python {platform.python_version()}")
say("adaptive query execution: " + spark.conf.get("spark.sql.adaptive.enabled"))

def sdiv(num, den):
    return F.when(den > 0, num / den)           # null unless the denominator is positive (same rule as Polars)

def spark_pipeline(df):
    c = F.col
    base = df.select("CERT", "qidx", *[c(x).cast("double").alias(x) for x in VALUE_COLS])
    base = base.select(
        "*",
        sdiv(c("EQ"), c("ASSET")).alias("eq_assets"),
        sdiv(c("NCLNLS"), c("LNLSGR")).alias("npl_loans"),
        sdiv(c("LNATRES"), c("LNLSGR")).alias("reserves_loans"),
        sdiv(c("LNLSGR"), c("ASSET")).alias("loans_assets"),
        sdiv(c("CHBAL") + c("SC"), c("ASSET")).alias("liquid_assets"),
        sdiv(c("BRO"), c("DEP")).alias("bro_dep"),
        F.when(c("ASSET") > 0, F.log(c("ASSET"))).alias("log_asset"),
    )
    # 4-quarter calendar lag: self-join on qidx + 4
    src = ["ASSET", "LNLSGR", "DEP", "eq_assets", "npl_loans"]
    lag = base.select("CERT", (c("qidx") + 4).alias("qidx"), *[c(x).alias(x + "_l4") for x in src])
    base = base.join(lag, ["CERT", "qidx"], "left")
    base = base.select(
        "*",
        F.when(c("ASSET_l4") > 0, c("ASSET") / c("ASSET_l4") - 1).alias("asset_yoy"),
        F.when(c("LNLSGR_l4") > 0, c("LNLSGR") / c("LNLSGR_l4") - 1).alias("loans_yoy"),
        F.when(c("DEP_l4") > 0, c("DEP") / c("DEP_l4") - 1).alias("dep_yoy"),
        (c("eq_assets") - c("eq_assets_l4")).alias("d_eq_assets_4q"),
        (c("npl_loans") - c("npl_loans_l4")).alias("d_npl_loans_4q"),
    )
    # rolling window over calendar quarters q-3 .. q per bank (range frame on qidx, so gaps never shift it)
    w4 = Window.partitionBy("CERT").orderBy("qidx").rangeBetween(-3, 0)
    base = base.select(
        "*",
        F.avg("ROAQ").over(w4).alias("roa_q_mean4"),
        F.stddev_samp("ROAQ").over(w4).alias("roa_q_std4"),
        F.count("ROAQ").over(w4).alias("roa_q_n4"),
    )
    # cross-section within the quarter: exact median (aggregate + broadcast join) and average-rank percentile
    med = base.groupBy("qidx").agg(
        F.expr("percentile(eq_assets, 0.5)").alias("eq_assets_med_q"),
        F.expr("percentile(npl_loans, 0.5)").alias("npl_loans_med_q"),
    )
    base = base.join(F.broadcast(med), "qidx", "left")
    nn = base.filter(c("eq_assets").isNotNull()).select("CERT", "qidx", "eq_assets")
    w_ord = Window.partitionBy("qidx").orderBy("eq_assets")
    rk = (nn.select("CERT", "qidx", F.rank().over(w_ord).alias("rk_min"),
                    F.count(F.lit(1)).over(Window.partitionBy("qidx", "eq_assets")).alias("ties"),
                    F.count(F.lit(1)).over(Window.partitionBy("qidx")).alias("n"))
            .select("CERT", "qidx", ((c("rk_min") + (c("ties") - 1) / 2.0) / c("n")).alias("eq_assets_pct_q")))
    base = base.join(rk, ["CERT", "qidx"], "left")
    return base.select("CERT", "qidx", *FEATURES).orderBy("CERT", "qidx")

sdf = spark.read.parquet(PATH)
say(f"input rows: {sdf.count():,}")

pl_res = polars_pipeline(pl.scan_parquet(PATH)).collect().to_pandas()
sp_res = spark_pipeline(sdf).toPandas()

def compare(a, b):
    keys_ok = a.shape == b.shape and bool((a[KEYS].to_numpy() == b[KEYS].to_numpy()).all())
    rows = []
    if keys_ok:
        for f in FEATURES:
            x = a[f].astype("float64").to_numpy(); y = b[f].astype("float64").to_numpy()
            nan_mis = int((np.isnan(x) != np.isnan(y)).sum()); both = ~np.isnan(x) & ~np.isnan(y)
            d = np.abs(x[both] - y[both]); atol = 1e-7 if f.endswith("_std4") else 1e-9
            ok = nan_mis == 0 and bool(np.allclose(x[both], y[both], rtol=1e-6, atol=atol))
            rows.append((f, float(d.max()) if d.size else 0.0, nan_mis, ok))
    return keys_ok, pd.DataFrame(rows, columns=["feature", "max_abs_diff", "null_mismatch", "match"])

keys_ok, tab = compare(pl_res, sp_res)
say(f"\nsame shape and (CERT, qidx) keys: {keys_ok} | rows {len(sp_res):,} | features {len(FEATURES)}")
say(tab.to_string(index=False))
say(f"\nfeatures matching Polars: {int(tab['match'].sum())} of {len(FEATURES)} | largest absolute difference: {tab['max_abs_diff'].max():.3g}")

['.config', 'part3_polars_pandas.py', 'spark_input.parquet', 'part3_polars_pandas.txt', 'sample_data']
machine: 2 logical CPUs, 13.6 GB RAM | spark 4.0.1 | pyspark 4.0.1 | polars 1.35.2 | python 3.13.18
adaptive query execution: true
input rows: 1,681,431

same shape and (CERT, qidx) keys: True | rows 1,681,431 | features 18
        feature  max_abs_diff  null_mismatch  match
      eq_assets  0.000000e+00              0   True
      npl_loans  0.000000e+00              0   True
 reserves_loans  0.000000e+00              0   True
   loans_assets  0.000000e+00              0   True
  liquid_assets  0.000000e+00              0   True
        bro_dep  0.000000e+00              0   True
      log_asset  3.552714e-15              0   True
      asset_yoy  0.000000e+00              0   True
      loans_yoy  0.000000e+00              0   True
        dep_yoy  0.000000e+00              0   True
 d_eq_assets_4q  0.000000e+00              0   True
 d_npl_loans_4q  0.000000e+00              0   Tr

In [ ]:
a = pl_res.set_index(["CERT", "qidx"])["roa_q_std4"]
b = sp_res.set_index(["CERT", "qidx"])["roa_q_std4"]
d = (a - b).abs().dropna()
for thr in (1e-12, 1e-9, 1e-8, 1e-7, 1e-6):
    print(f"rows with |diff| > {thr:g}: {(d > thr).sum():,} of {len(d):,}")

bad = d[d > 1e-7].sort_values(ascending=False)
raw = pd.read_parquet(PATH, columns=["CERT", "qidx", "ROAQ"])
raw["ROAQ"] = raw["ROAQ"].astype("float64")
raw = raw[raw["CERT"].isin({c for c, _ in bad.index})]
groups = {c: g for c, g in raw.groupby("CERT")}

err_pl, err_sp = [], []
for i, ((cert, q), diff) in enumerate(bad.items()):
    g = groups[cert]
    w = g[(g.qidx >= q - 3) & (g.qidx <= q)]["ROAQ"].dropna().to_numpy()
    exact = np.std(w, ddof=1) if len(w) > 1 else np.nan
    err_pl.append(abs(a.loc[(cert, q)] - exact)); err_sp.append(abs(b.loc[(cert, q)] - exact))
    if i < 4:
        print(f"\nCERT {cert}, qidx {q}: window {w.tolist()} | exact {exact!r} | polars {a.loc[(cert, q)]!r} | spark {b.loc[(cert, q)]!r}")
if len(bad):
    print(f"\nrows above 1e-7: {len(bad)} | max error vs exact: polars {np.nanmax(err_pl):.3g}, spark {np.nanmax(err_sp):.3g}")
    print("which engine is closer to the exact value in these rows: "
          f"polars {int((np.array(err_pl) < np.array(err_sp)).sum())}, spark {int((np.array(err_sp) < np.array(err_pl)).sum())}")

rows with |diff| > 1e-12: 287 of 1,652,719
rows with |diff| > 1e-09: 49 of 1,652,719
rows with |diff| > 1e-08: 39 of 1,652,719
rows with |diff| > 1e-07: 15 of 1,652,719
rows with |diff| > 1e-06: 0 of 1,652,719

CERT 27229, qidx 7999: window [0.0, 0.0, 0.0, 0.0] | exact np.float64(0.0) | polars np.float64(6.743495761743046e-07) | spark np.float64(0.0)

CERT 27229, qidx 8000: window [0.0, 0.0, 0.0, 0.0] | exact np.float64(0.0) | polars np.float64(6.743495761743046e-07) | spark np.float64(0.0)

CERT 27229, qidx 8001: window [0.0, 0.0, 0.0, 0.0] | exact np.float64(0.0) | polars np.float64(6.743495761743046e-07) | spark np.float64(0.0)

CERT 27229, qidx 8002: window [0.0, 0.0, 0.0, 0.0] | exact np.float64(0.0) | polars np.float64(6.743495761743046e-07) | spark np.float64(0.0)

rows above 1e-7: 15 | max error vs exact: polars 6.74e-07, spark 0
which engine is closer to the exact value in these rows: polars 0, spark 15


In [ ]:
def compare_tol(a, b, tol_std):
    rows = []
    for f in FEATURES:
        x = a[f].astype("float64").to_numpy(); y = b[f].astype("float64").to_numpy()
        nan_mis = int((np.isnan(x) != np.isnan(y)).sum()); both = ~np.isnan(x) & ~np.isnan(y)
        d = np.abs(x[both] - y[both]); atol = tol_std if f.endswith("_std4") else 1e-9
        ok = nan_mis == 0 and bool(np.allclose(x[both], y[both], rtol=1e-6, atol=atol))
        rows.append((f, float(d.max()) if d.size else 0.0, nan_mis, ok))
    return pd.DataFrame(rows, columns=["feature", "max_abs_diff", "null_mismatch", "match"])

tab2 = compare_tol(pl_res, sp_res, 1e-6)
say("\n== Diagnosis of roa_q_std4 (Spark vs Polars) ==")
say(f"rows with |diff| > 1e-7: {len(bad)} of {len(d):,}; none above 1e-6 (largest {d.max():.3g})")
say(f"worst row: CERT {bad.index[0][0]}, qidx {bad.index[0][1]}: window of four ROAQ values all 0.0, exact std = 0, "
    f"polars {a.loc[bad.index[0]]:.3g}, spark {b.loc[bad.index[0]]:.3g}")
say(f"max error vs the exact value (NumPy, two-pass) in those rows: polars {np.nanmax(err_pl):.3g}, spark {np.nanmax(err_sp):.3g}")
say(f"engine closer to the exact value: spark in {int((np.array(err_sp) < np.array(err_pl)).sum())} of {len(bad)} rows")
say("\n== Re-evaluation with tolerance 1e-6 for roa_q_std4 (1e-9 for the other features) ==")
say(tab2.to_string(index=False))
say(f"features matching Polars: {int(tab2['match'].sum())} of {len(FEATURES)} | largest absolute difference: {tab2['max_abs_diff'].max():.3g}")


== Diagnosis of roa_q_std4 (Spark vs Polars) ==
rows with |diff| > 1e-7: 15 of 1,652,719; none above 1e-6 (largest 6.74e-07)
worst row: CERT 27229, qidx 7999: window of four ROAQ values all 0.0, exact std = 0, polars 6.74e-07, spark 0
max error vs the exact value (NumPy, two-pass) in those rows: polars 6.74e-07, spark 0
engine closer to the exact value: spark in 15 of 15 rows

== Re-evaluation with tolerance 1e-6 for roa_q_std4 (1e-9 for the other features) ==
        feature  max_abs_diff  null_mismatch  match
      eq_assets  0.000000e+00              0   True
      npl_loans  0.000000e+00              0   True
 reserves_loans  0.000000e+00              0   True
   loans_assets  0.000000e+00              0   True
  liquid_assets  0.000000e+00              0   True
        bro_dep  0.000000e+00              0   True
      log_asset  3.552714e-15              0   True
      asset_yoy  0.000000e+00              0   True
      loans_yoy  0.000000e+00              0   True
        dep_yo

In [ ]:
def plan_text(mode):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        spark_pipeline(sdf).explain(mode=mode)
    return buf.getvalue()

formatted, extended = plan_text("formatted"), plan_text("extended")
open("/content/part3_spark_plan.txt", "w").write("== FORMATTED PHYSICAL PLAN ==\n" + formatted + "\n\n== EXTENDED (parsed, analysed, optimised, physical) ==\n" + extended)
say("\n== Plan facts (formatted physical plan) ==")
for name, pat in [("Exchange (shuffle)", r"\bExchange\b"), ("SortMergeJoin", r"SortMergeJoin"),
                  ("BroadcastHashJoin", r"BroadcastHashJoin"), ("Window", r"\bWindow\b"),
                  ("Sort", r"\bSort\b"), ("FileScan parquet", r"Scan parquet")]:
    say(f"{name:<22} {len(re.findall(pat, formatted))}")
rs = re.findall(r"ReadSchema: struct<([^>]*)>", formatted)
say("columns read by each scan: " + ", ".join(str(len(r.split(","))) for r in rs) + " of 12 in the input file")
print(formatted[:6000])


== Plan facts (formatted physical plan) ==
Exchange (shuffle)     24
SortMergeJoin          8
BroadcastHashJoin      2
Window                 8
Sort                   24
FileScan parquet       12
columns read by each scan: 12, 7, 6, 2, 4, 2 of 12 in the input file
== Physical Plan ==
AdaptiveSparkPlan (66)
+- Sort (65)
   +- Exchange (64)
      +- Project (63)
         +- SortMergeJoin LeftOuter (62)
            :- Sort (38)
            :  +- Exchange (37)
            :     +- Project (36)
            :        +- BroadcastHashJoin LeftOuter BuildRight (35)
            :           :- Project (17)
            :           :  +- Window (16)
            :           :     +- Sort (15)
            :           :        +- Exchange (14)
            :           :           +- Project (13)
            :           :              +- SortMergeJoin LeftOuter (12)
            :           :                 :- Sort (5)
            :           :                 :  +- Exchange (4)
            :          

In [ ]:
say("\n== Plan facts, each node counted once (details section of the formatted plan) ==")
for name in ["Scan parquet", "Exchange", "BroadcastExchange", "Sort", "Window", "SortMergeJoin",
             "BroadcastHashJoin", "ObjectHashAggregate"]:
    say(f"{name:<22} {len(re.findall(rf'(?m)^\(\d+\) {name}\b', formatted))}")


== Plan facts, each node counted once (details section of the formatted plan) ==
Scan parquet           6
Exchange               12
BroadcastExchange      1
Sort                   12
Window                 4
SortMergeJoin          4
BroadcastHashJoin      1
ObjectHashAggregate    2


In [ ]:
import time
import numpy as np

results = []
parts_list = [4, 8, 16, 32, 64]

for parts in parts_list:
    spark.conf.set("spark.sql.shuffle.partitions", str(parts))

    times = []

    # Chạy 3 lần đo
    for _ in range(3):
        start = time.perf_counter()

        spark_pipeline(sdf).count()  # thực thi Spark

        elapsed = time.perf_counter() - start
        times.append(elapsed)

    best = min(times)
    median = np.median(times)

    results.append((parts, best, median))

    print(
        f"Partitions: {parts:>2} | "
        f"Best: {best:.2f}s | "
        f"Median: {median:.2f}s"
    )

Partitions:  4 | Best: 9.20s | Median: 9.32s
Partitions:  8 | Best: 8.40s | Median: 10.66s
Partitions: 16 | Best: 9.58s | Median: 10.46s
Partitions: 32 | Best: 10.56s | Median: 10.69s
Partitions: 64 | Best: 9.85s | Median: 10.94s


In [ ]:
def timed(fn, repeat=3):
    fn()                                   # warm-up (JIT, file cache)
    ts = []
    for _ in range(repeat):
        t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return ts

t_pl = timed(lambda: polars_pipeline(pl.scan_parquet(PATH)).collect())
t_sp = timed(lambda: spark_pipeline(sdf).write.format("noop").mode("overwrite").save())  # runs everything, no collect
say("\n== Time on the same Colab machine (best / median of 3, after 1 warm-up) ==")
say(f"polars lazy : {min(t_pl):.2f} s / {np.median(t_pl):.2f} s")
say(f"spark local : {min(t_sp):.2f} s / {np.median(t_sp):.2f} s  (output not collected to the driver)")
say(f"polars is {min(t_sp)/min(t_pl):.1f}x faster than Spark on this data size")


== Time on the same Colab machine (best / median of 3, after 1 warm-up) ==
polars lazy : 2.89 s / 3.02 s
spark local : 52.51 s / 55.37 s  (output not collected to the driver)
polars is 18.2x faster than Spark on this data size


In [ ]:
from functools import reduce

def rep_pl(lf, k):
    return pl.concat([lf.with_columns((pl.col("CERT") + i * 10_000_000).alias("CERT")) for i in range(k)])

def rep_sp(df, k):
    return reduce(DataFrame.unionByName, [df.withColumn("CERT", F.col("CERT") + i * 10_000_000) for i in range(k)])

base_rows = sdf.count()
say("\n== Scaling on Colab (1 run per size, after the warm-up above) ==")
say("engine | size | rows | seconds")
for k in [1, 2]:
    t = time.perf_counter(); n = len(polars_pipeline(rep_pl(pl.scan_parquet(PATH), k)).collect()); dt = time.perf_counter() - t
    say(f"polars | {k}x | {n:,} | {dt:.1f}")
for k in [1, 2, 4]:
    t = time.perf_counter(); spark_pipeline(rep_sp(sdf, k)).write.format("noop").mode("overwrite").save(); dt = time.perf_counter() - t
    say(f"spark  | {k}x | {base_rows * k:,} | {dt:.1f}")

open("/content/part3_spark_results.txt", "w").write("\n".join(log))
files.download("/content/part3_spark_results.txt")
files.download("/content/part3_spark_plan.txt")


== Scaling on Colab (1 run per size, after the warm-up above) ==
engine | size | rows | seconds
polars | 1x | 1,681,431 | 3.8
polars | 2x | 3,362,862 | 8.2
spark  | 1x | 1,681,431 | 55.4
spark  | 2x | 3,362,862 | 120.6
spark  | 4x | 6,725,724 | 244.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls -lh /content/part3_spark_*

-rw-r--r-- 1 root root  98K Nov 23 04:12 /content/part3_spark_plan.txt
-rw-r--r-- 1 root root 3.8K Nov 23 04:27 /content/part3_spark_results.txt


In [ ]:
import time

def timed_noop(repeat=2):
    spark_pipeline(sdf).write.format("noop").mode("overwrite").save()      # warm-up
    ts = []
    for _ in range(repeat):
        t = time.perf_counter()
        spark_pipeline(sdf).write.format("noop").mode("overwrite").save()
        ts.append(time.perf_counter() - t)
    return ts

lines = ["== Spark shuffle partitions (same 18 features, Colab 2 vCPU, one warm-up then two timed runs) =="]
for aqe, n in [("true", 8), ("true", 200), ("false", 200)]:
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    spark.conf.set("spark.sql.shuffle.partitions", str(n))
    ts = timed_noop()
    lines.append(f"adaptive={aqe} shuffle.partitions={n}: runs {ts[0]:.1f} s and {ts[1]:.1f} s")
    print(lines[-1])
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "16")
print("report used partitions=16, adaptive=true: best of 3 = 52.51 s (run 2)")

adaptive=true shuffle.partitions=8: runs 72.6 s and 59.8 s
adaptive=true shuffle.partitions=200: runs 63.4 s and 59.1 s
adaptive=false shuffle.partitions=200: runs 96.3 s and 107.8 s
report used partitions=16, adaptive=true: best of 3 = 52.51 s (run 2)
